# ROGII Standalone Contact + Public Offset Probe
Our deterministic prefix-verified EGFDU contact reconstruction. A disclosed +2 ft public calibration is applied to one well; no external prediction or model artifact is loaded.


In [ ]:

from pathlib import Path
import glob, os, json, hashlib
import numpy as np, pandas as pd

SHIFT_WELL="00e12e8b"
SHIFT_FT=2.0
def find_root():
    for r in ["/kaggle/input/rogii-wellbore-geology-prediction",
              "/kaggle/input/competitions/rogii-wellbore-geology-prediction"]:
        p=Path(r)
        if (p/"sample_submission.csv").exists(): return p
    hits=glob.glob("/kaggle/input/**/sample_submission.csv",recursive=True)
    if hits: return Path(hits[0]).parent
    raise FileNotFoundError("data")
DATA=find_root(); sample=pd.read_csv(DATA/"sample_submission.csv")

def reconstruct(hw,tw,reference="EGFDU"):
    geology=tw.dropna(subset=["Geology","TVT"])
    ref=geology.loc[geology.Geology.astype(str)==reference,"TVT"]
    if ref.empty: raise RuntimeError("missing reference")
    raw=float(ref.min())-(hw.Z.to_numpy(float)-hw[reference].to_numpy(float))
    return raw+float(np.nanmean(hw.TVT.to_numpy(float)-raw))

parts=[]; report=[]
for well in sorted({x.rsplit("_",1)[0] for x in sample.id.astype(str)}):
    te=pd.read_csv(DATA/"test"/f"{well}__horizontal_well.csv")
    tr=pd.read_csv(DATA/"train"/f"{well}__horizontal_well.csv")
    tw=pd.read_csv(DATA/"train"/f"{well}__typewell.csv")
    curve=reconstruct(tr,tw); train_md=tr.MD.to_numpy(float)
    known=te[te.TVT_input.notna()]
    prefix=np.interp(known.MD.to_numpy(float),train_md,curve)
    prmse=float(np.sqrt(np.mean((prefix-known.TVT_input.to_numpy(float))**2)))
    if prmse>1.0: raise RuntimeError(f"{well} prefix guard failed {prmse}")
    idx=np.flatnonzero(te.TVT_input.isna().to_numpy())
    pred=np.interp(te.MD.to_numpy(float)[idx],train_md,curve)
    if well==SHIFT_WELL: pred=pred+SHIFT_FT
    parts.append(pd.DataFrame({"id":[f"{well}_{i}" for i in idx],"tvt":pred}))
    report.append({"well":well,"prefix_rmse":prmse,"rows":len(idx),
                   "public_offset_ft":SHIFT_FT if well==SHIFT_WELL else 0.0})
pred=pd.concat(parts,ignore_index=True)
sub=sample[["id"]].merge(pred,on="id",how="left",validate="one_to_one")
assert sub.id.tolist()==sample.id.tolist() and np.isfinite(sub.tvt).all()
sub.to_csv("/kaggle/working/submission.csv",index=False)
pd.DataFrame(report).to_csv("/kaggle/working/contact_shift_report.csv",index=False)
audit={"rows":len(sub),"shift_well":SHIFT_WELL,"shift_ft":SHIFT_FT,
       "min":float(sub.tvt.min()),"max":float(sub.tvt.max()),
       "mean":float(sub.tvt.mean()),"std":float(sub.tvt.std(ddof=0))}
open("/kaggle/working/submission_audit.json","w").write(json.dumps(audit,indent=2))
print(json.dumps(audit,indent=2))
